In [1]:
import pandas as pd

data = pd.read_csv("Full Compiled Data.csv").set_index("POSTING ID")
text = pd.read_csv("/data/reddit/all_posts_w_deleted_age_gender.csv").set_index("post_id")
text

/tmp/ipykernel_30472/1367500409.py:4: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  text = pd.read_csv("/data/reddit/all_posts_w_deleted_age_gender.csv").set_index("post_id")


,title,url,author,score,num_comments,created_utc,comments_url,selftext,retrieved_on,subreddit,deleted_post,mod_post,all_text,gender,age
post_id,,,,,,,,,,,,,,,
fqo2sz,38[M]. Its a beautiful day in Florida for a dr...,https://i.redd.it/1j65ikp74gp41.jpg,another_day_in_fl,2.0,0,1585415322,/r/FreeCompliments/comments/fqo2sz/38m_its_a_b...,NaN,1.585416e+09,freecompliments,False,False,38[M]. Its a beautiful day in Florida for a dr...,m,38.0
fqny0d,Sometimes there are days when you are on top o...,https://i.redd.it/mflfp62z2gp41.jpg,thirsk77,1.0,0,1585414907,/r/FreeCompliments/comments/fqny0d/sometimes_t...,NaN,1.585416e+09,freecompliments,False,False,Sometimes there are days when you are on top o...,NaN,NaN
fqnqy6,Hey guys! Here's a poem I've been working on. ...,https://i.redd.it/9su636uy0gp41.png,PM_UR_PUPPER,1.0,0,1585414246,/r/FreeCompliments/comments/fqnqy6/hey_guys_he...,NaN,1.585415e+09,freecompliments,False,False,Hey guys! Here's a poem I've been working on. ...,NaN,NaN
fqn8zc,"24(M), recently dumped out no where, could use...",https://i.redd.it/gqqeazk1wfp41.jpg,Jmcguitar,1.0,9,1585412575,/r/FreeCompliments/comments/fqn8zc/24m_recentl...,NaN,1.585413e+09,freecompliments,False,False,"24(M), recently dumped out no where, could use...",m,24.0
fqn0zm,"21M - My heart feels super heavy, had been tal...",https://i.redd.it/ivq02fwutfp41.jpg,INTLHelper,1.0,0,1585411841,/r/FreeCompliments/comments/fqn0zm/21m_my_hear...,NaN,1.585412e+09,freecompliments,False,False,"21M - My heart feels super heavy, had been tal...",m,21.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6u9qug,https://m.imgur.com/a/0VTj6,http://[M20] Rate Me,[deleted],1.0,0,1502974366,/r/truerateme/comments/6u9qug/httpsmimgurcoma0...,[deleted],1.503056e+09,truerateme,True,False,\n[deleted],NaN,NaN
6u9dkw,"[M, 32, 6""1] Rate me!",https://i.redd.it/kn6rd52bbagz.png,throwitawaythekey,9.0,40,1502969875,/r/truerateme/comments/6u9dkw/m_32_61_rate_me/,NaN,1.503051e+09,truerateme,False,False,"[M, 32, 6""1] Rate me!",NaN,NaN
6u7jht,Rating system guideline,https://www.reddit.com/r/truerateme/comments/6...,kalkush,7.0,1,1502942796,/r/truerateme/comments/6u7jht/rating_system_gu...,[removed],1.503024e+09,truerateme,False,False,Rating system guideline\n[removed],NaN,NaN


In [2]:
desired_columns = ["Physical transformation","Future self","Past self","Current self","Physical wishes"]
data = data[desired_columns]

In [3]:
data['Looks/Body'] = data[desired_columns].astype(str).agg(' '.join, axis=1)
data['Looks/Body'] = data['Looks/Body'].apply(lambda x: 1 if '1' in x else 0)

In [4]:
desired_columns = ["Looks/Body"]
data = data[desired_columns]

In [5]:
desired_columns = ["all_text"]
text = text.rename_axis("POSTING ID")
text = text[desired_columns]

In [6]:
data.rename(columns={'post_id': 'POSTING ID'}, inplace=True)

In [7]:
df = pd.merge(text, data, on='POSTING ID')
df.rename(columns={'Looks/Body': 'label'}, inplace=True)

In [8]:
df.reset_index(drop=True)

,all_text,label
0,Been having a really rough week and feeling do...,0
1,I could have saved a suicide victim. I still f...,0
2,I was told I'm to skinny. But when I was heavi...,0
3,Just having a great day today! Hope y’all are ...,0
4,Just felt cute today and sometimes sharing a p...,0
...,...,...
1250,26(5'9) Shoot.,1
1251,"26M FA, is it over my dudes?\n[deleted]",0
1252,The more honest the better.\n[deleted],0
1253,Honest rate please? This is my 3rd time postin...,1


In [9]:
import logging
import datasets
import matplotlib.pyplot as plt
import gc
import torch
from sklearn.metrics import precision_score, accuracy_score, f1_score, classification_report, PrecisionRecallDisplay
import numpy as np
import os
from small_text.base import LABEL_IGNORED

from small_text import (
    EmptyPoolException,
    PoolBasedActiveLearner,
    PoolExhaustedException,
    BreakingTies,
    EmbeddingKMeans,
    SetFitClassificationFactory,
    SetFitModelArguments,
    TextDataset,
    random_initialization_balanced,
    SubsamplingQueryStrategy
)

import pandas as pd

import pigeonXT as pixt
from typing import List, Optional, Any, Tuple, Dict

class Annotation:
    @staticmethod
    def run_annotation(df: pd.DataFrame, labels: List[str], column_name: str)->pixt.annotate:
        # This will only setup the annotation, and needs to be confirmed via UI interaction
        return pixt.annotate(
            examples=df[[column_name]].rename(columns={column_name: 'example'}),
            options=labels,
            task_type='classification',
            buttons_in_a_row=3,
            reset_buttons_after_click=True,
            include_next=True
        )
    
def samp(dat,n):
    if len(dat) < n:
        return dat.drop(columns=["label"])
    return dat.sample(n).drop(columns=["label"])
    
# disables the progress bar for notebooks: https://github.com/huggingface/datasets/issues/2651
datasets.logging.get_verbosity = lambda: logging.NOTSET

POSSIBLE_LABELS = [0,1]

target_labels = np.arange(len(POSSIBLE_LABELS)-1).astype(int)

2024-10-26 17:59:37.725880: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-26 17:59:37.815467: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-26 17:59:37.839514: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-26 17:59:37.998271: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-26 17:59:39.077145: W tensorflow/compiler/tf2

In [10]:
all_unlabeled = text.iloc[1000:1200]

In [11]:
init_labels = np.array([1 if s == '1' else 0 for s in df.label.values.tolist()])
init_dataset = TextDataset.from_arrays(df.all_text.values.tolist(),                        
                                        init_labels,
                                        target_labels=target_labels)

/home/mrbienie/.local/lib/python3.9/site-packages/small_text/utils/annotations.py:67: ExperimentalWarning: The function from_arrays is experimental and maybe subject to change soon.
  warnings.warn(f'The {subject} {func_or_class.__name__} is experimental '
/home/mrbienie/.local/lib/python3.9/site-packages/small_text/data/datasets.py:597: DeprecationWarning: The function get_flattened_unique_labels has been deprecated in 1.1.0 and will be removed in 2.0.0.
  encountered_labels = get_flattened_unique_labels(self)


In [12]:
full_labeled_sample = df[['all_text','label']].rename(columns={"all_text":"modeling_text"})
full_labeled_sample

,modeling_text,label
POSTING ID,,
f4hdxa,Been having a really rough week and feeling do...,0
f4duvi,I could have saved a suicide victim. I still f...,0
f1nfyk,I was told I'm to skinny. But when I was heavi...,0
exvzcv,Just having a great day today! Hope y’all are ...,0
epqeqh,Just felt cute today and sometimes sharing a p...,0
...,...,...
7ip58w,26(5'9) Shoot.,1
7cpll2,"26M FA, is it over my dudes?\n[deleted]",0
792qvh,The more honest the better.\n[deleted],0


In [13]:
all_unlabeled = all_unlabeled[~all_unlabeled.all_text.isin(full_labeled_sample.modeling_text.values.tolist())]

/home/kjoseph/miniconda3/lib/python3.9/site-packages/pandas/core/algorithms.py:516: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  common = np.find_common_type([values.dtype, comps.dtype], [])


In [14]:
full_labeled_sample
full_labeled_sample_ones = full_labeled_sample[full_labeled_sample['label'] == 1]
full_labeled_sample_zeros = full_labeled_sample[full_labeled_sample['label'] == 0]
sampled_ones = full_labeled_sample_ones.sample(n=50, random_state=42)
sampled_zeros = full_labeled_sample_zeros.sample(n=50, random_state=42)
balanced_df = pd.concat([sampled_ones, sampled_zeros])
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [15]:
validation_data = balanced_df

In [16]:
init_labeled = full_labeled_sample.groupby("label").apply(samp,n=200).reset_index()
#validation_data = full_labeled_sample[~full_labeled_sample.modeling_text.isin(init_labeled.modeling_text)].groupby("label").apply(samp,n=100).reset_index()
len(init_labeled), len(validation_data)

(365, 100)

In [17]:
validation_data['label'].nunique() 

2

In [18]:
target_labels = [0,1]

In [19]:
init_dataset = TextDataset.from_arrays(init_labeled.modeling_text.values.tolist(),                        
                                       init_labeled.label.values,
                                       target_labels=target_labels)

/home/mrbienie/.local/lib/python3.9/site-packages/small_text/utils/annotations.py:67: ExperimentalWarning: The function from_arrays is experimental and maybe subject to change soon.
  warnings.warn(f'The {subject} {func_or_class.__name__} is experimental '
/home/mrbienie/.local/lib/python3.9/site-packages/small_text/data/datasets.py:597: DeprecationWarning: The function get_flattened_unique_labels has been deprecated in 1.1.0 and will be removed in 2.0.0.
  encountered_labels = get_flattened_unique_labels(self)


In [20]:
print("Labels in data:", init_labeled.label.unique())
print("Target labels:", target_labels)

Labels in data: [0 1]
Target labels: [0, 1]


In [21]:
validation_dataset = TextDataset.from_arrays(validation_data.modeling_text.values.tolist(),                        
                                             validation_data.label.values,
                                               target_labels=target_labels)

In [22]:
init_labeled[init_labeled.label ==1 ].sample(3)

,label,POSTING ID,modeling_text
332,1,3lctk5,[19m] bad skin but please be honest\n[deleted]
270,1,4j3ur6,[24M] Babyface and a Five Head. I'm tired of t...
266,1,5ctmyh,"23 yo guy 6 months ago, new post soon [1\2]\n[..."


In [23]:
posts = all_unlabeled.all_text.values.tolist()

In [24]:
len(init_dataset.x)

365

In [25]:
from small_text import LABEL_UNLABELED
model_args = SetFitModelArguments('sentence-transformers/paraphrase-mpnet-base-v2')

clf_factory = SetFitClassificationFactory(model_args,
                                          len(POSSIBLE_LABELS),
                                          classification_kwargs=dict({
                                              'device': 'cuda',
                                              'mini_batch_size': 8
                                          }))

# define a query strategy and initialize a pool-based active learner
query_strategy = SubsamplingQueryStrategy(BreakingTies())

def initialize_with_warmstart(init_dataset):
    

    # Append the initial labeled data to our train dataset. This is only necessary because the logistic regression head 
    #   implicitly obtains the number of classes from the training data. If we omitted this and the first query 
    #   would not return all four labels, the model head would predict three classes instead of four.
    labeled_indices = np.arange(len(init_dataset.y))

    train = TextDataset.from_arrays(init_dataset.x + posts, 
                                    np.append(init_dataset.y, np.array([LABEL_UNLABELED]*len(posts))), 
                                    target_labels=target_labels)
    
    # suppress progress bars in jupyter notebook
    setfit_train_kwargs = {'show_progress_bar': False}

    active_learner = PoolBasedActiveLearner(clf_factory, query_strategy, train, 
                                            fit_kwargs={'setfit_train_kwargs': setfit_train_kwargs})
    active_learner._clf = clf_factory.new()
    active_learner._clf.fit(init_dataset, setfit_train_kwargs=setfit_train_kwargs)

    active_learner.y = init_dataset.y
    active_learner.indices_labeled = labeled_indices
    active_learner._index_to_position = active_learner._build_index_to_position_dict()
    
    return active_learner, train


active_learner, train = initialize_with_warmstart(init_dataset)

/home/mrbienie/.local/lib/python3.9/site-packages/small_text/utils/annotations.py:67: ExperimentalWarning: The function from_arrays is experimental and maybe subject to change soon.
  warnings.warn(f'The {subject} {func_or_class.__name__} is experimental '
/home/mrbienie/.local/lib/python3.9/site-packages/small_text/data/datasets.py:597: DeprecationWarning: The function get_flattened_unique_labels has been deprecated in 1.1.0 and will be removed in 2.0.0.
  encountered_labels = get_flattened_unique_labels(self)


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call,so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.

In [ ]:
def evaluate(active_learner, train, test):
    
    if len(train) == 0:
        return np.nan
    
    y_pred = active_learner.classifier.predict(train)
    y_score = active_learner.classifier.predict_proba(test)
    y_pred_test = active_learner.classifier.predict(test)
    
    test_acc = accuracy_score(y_pred_test, test.y)
    test_f1 = f1_score(test.y, y_pred_test, average="macro")

    print('Train accuracy: {:.2f}'.format(accuracy_score(y_pred, train.y)))
    print('Test accuracy: {:.2f}'.format(test_acc))
    print('Test F1: {:.2f}'.format(test_f1))
    print(classification_report(test.y,y_pred_test))
    return test_acc


results_setfit = []
results_setfit.append(evaluate(active_learner, train[active_learner.indices_labeled], validation_dataset))

In [26]:
round_v = 1
prefix = "Looks_Body"

In [27]:
train_dat = pd.DataFrame({"tr":train.x})

NameError: name 'train' is not defined

In [63]:
num_queries = 50

round_v = round_v + 1
# ...where each iteration consists of labelling 20 samples
q_indices = active_learner.query(num_samples=num_queries)

annotations = Annotation.run_annotation(train_dat.iloc[q_indices,:],['0','1'],'tr')

HTML(value='0 of 50 Examples annotated, Current Position: 0 ')

Output()

/home/kjoseph/miniconda3/lib/python3.9/site-packages/pandas/core/dtypes/cast.py:1841: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])


In [75]:
out_fil = f"ann_{prefix}_{round_v}.csv"
if os.path.exists(out_fil):
    print("file exists, change the name")
else:
    annotations.to_csv(out_fil,index=False)

In [77]:
def set_labels(annotations):
    labels = []
    for x in annotations.label:
        if x == '1':
            labels.append(1)
        elif x == '0':
            labels.append(0)
        else:
            labels.append(LABEL_IGNORED)
        
        #lab = get_val(x)
        # if lab is None:
        #     labels.append(LABEL_IGNORED)
        # else:
        #     labels.append(lab)
    return np.array(labels)

In [78]:
active_learner.update(set_labels(annotations))

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/home/mrbienie/.local/lib/python3.9/site-packages/small_text/data/datasets.py:597: DeprecationWarning: The function get_flattened_unique_labels has been deprecated in 1.1.0 and will be removed in 2.0.0.
  encountered_labels = get_flattened_unique_labels(self)
/home/mrbienie/.local/lib/python3.9/site-packages/small_text/integrations/transformers/classifiers/setfit.py:261: DeprecationWarning: `SetFitTrainer` has been deprecated and will be removed in v2.0.0 of SetFit. Please use `Trainer` instead.
  trainer = SetFitTrainer(


Map:   0%|          | 0/415 [00:00<?, ? examples/s]

/home/mrbienie/.local/lib/python3.9/site-packages/small_text/integrations/transformers/classifiers/setfit.py:269: DeprecationWarning: `SetFitTrainer.train` does not accept keyword arguments anymore. Please provide training arguments via a `TrainingArguments` instance to the `SetFitTrainer` initialisation or the `SetFitTrainer.train` method.
  trainer.train(max_length=self.max_seq_len, **setfit_train_kwargs)
***** Running training *****
  Num unique pairs = 16600
  Batch size = 8
  Num epochs = 1
  Total optimization steps = 2075


Step,Training Loss



# memory fix: https://github.com/UKPLab/sentence-transformers/issues/487, https://github.com/UKPLab/sentence-transformers/issues/1793
gc.collect()
torch.cuda.empty_cache()

print('---------------')
print('Iteration #{:d} ({} samples)'.format(0, len(active_learner.indices_labeled)))
results_setfit.append(evaluate(active_learner, train[active_learner.indices_labeled], validation_dataset))

In [85]:
pd.set_option('display.max_colwidth', None)
m = pd.DataFrame({"x":validation_dataset.x, "y":validation_dataset.y, "pred": active_learner.classifier.predict_proba(validation_dataset)[:,1]})
m['pred_bin'] = m['pred'] > .5
m[(m.pred_bin == 0) & (m.y == 0)]

,x,y,pred,pred_bin
0,21M Rate and CC!\n[deleted],0,0.001623,False
2,"M 22 | Sort of always been insecure, regardless of that I'd just like peoples' honest opinion of how I might be able to improve my looks\n[deleted]",0,0.001642,False
7,"Couldn't help a friend when she felt suicidal today, she didn't do anything but i feel like absolute shit, Pre-Work selfie",0,0.001565,False
12,(F18) Last time I posted here I admittedly was pretty selective on what to post because who isn't but I tried being a bit more 'honest' this time,0,0.001525,False
14,Be kind\n[removed],0,0.001689,False
16,Lost as. Curious on opinions all honestly welcome,0,0.001673,False
17,24M what's the verdict? I have no idea where I stand in terms of looks.,0,0.001579,False
20,"M23. Grew up with self-esteem issues, working to better myself and could use some feedback. First 3 pictures are a few years old.",0,0.001561,False
21,What should I do with my hair?\nI did just come from playing a basketball so my hair isn't exactly in pristine condition... Thanks in advance\n\n\n,0,0.018997,False
24,Do your worst (: ( F)(20)\n[removed],0,0.001579,False
